# 유니버스 → 전략 판단 → 원장

저장된 S&P 500 일봉으로 블록 그래프를 만들고, 한 시점 실행과 1년 백테스트를 차례로 본다.
모든 블록은 역할 하나만 가진다. 팩터 갈래(점수 → 선정 → 비중)와 시그널 갈래(조건 → 이벤트)가
같은 단계 띠를 쓰고, 둘 다 **전략 판단(STRATEGY) → 주문 실행(EXECUTION)** 으로 끝난다.

커널은 `.venv/bin/python`을 고른다. 구조 설명은 `docs/ARCHITECTURE.md` 6장에 있다.

In [ ]:
import sys
from dataclasses import fields, is_dataclass
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "ai_studio").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


def table(value):
    """프레임·전략 판단·이력 튜플을 표로."""
    rows = getattr(value, "rows", None) or getattr(value, "signals", None) or getattr(value, "positions", None)
    if rows is None:
        rows = value if isinstance(value, (list, tuple)) else (value,)
    if not rows:
        return pd.DataFrame()
    if is_dataclass(rows[0]):
        return pd.DataFrame([{f.name: getattr(row, f.name) for f in fields(rows[0])} for row in rows])
    return pd.DataFrame(rows)

from ai_studio.n02_schema import STAGES, StrategyEdge, StrategyGraph, StrategyNode
from ai_studio.n03_market_data import history_from_bars, load_ohlc
from ai_studio.n10_functions import FUNCTIONS, get_function
from ai_studio.n12_strategies import EXAMPLES, PRESETS, example_momentum, example_momentum_gated_cross, place
from ai_studio.n13_connect import validate_strategy
from ai_studio.n14_engine import ExecutionContext, execute, execution_order
from ai_studio.n15_ohlc_backtest import RunSettings, run_graph_backtest

## 1. 시세 적재

일봉은 블록 사이로 흐르는 프레임이 아니라 실행 맥락이다. `ExecutionContext`로 넘긴다.
시각은 뉴욕 거래소 기준이다. 봉의 `timestamp`는 그날 16:00 종가 시각, `open_at`은 09:30 시가 시각이다.

In [ ]:
bars = load_ohlc()
history = history_from_bars(bars)
as_of = max(bar.timestamp for bar in bars)
print(f"종목 {len(history):,}  봉 {len(bars):,}  기간 {min(b.timestamp for b in bars):%Y-%m-%d} ~ {as_of:%Y-%m-%d %H:%M %Z}")

## 2. 단계와 블록

블록의 역할(role)이 캔버스 단계를 정한다. 단계 하나에 여러 역할이 들어갈 수 있지만, 블록 하나가 두 역할을 갖지는 않는다.

In [ ]:
pd.DataFrame([
    {"단계": label, "역할": role, "블록": ", ".join(fn.id for fn in FUNCTIONS if fn.role == role)}
    for _, label, roles in STAGES for role in roles
])

## 3. 프리셋 — 자주 쓰는 조합

프리셋은 블록이 아니라 저장된 하위 그래프다. 펼치면 안쪽 블록이 그대로 놓이고 파라미터를 따로 고칠 수 있다.
예를 들어 골든크로스는 이동평균 두 개 → 지표 비교 → 상태 전이다.

In [ ]:
pd.DataFrame([{"프리셋": p.name, "블록": " → ".join(n.function_id for n in p.nodes), "출력": ", ".join(k for k, _, _ in p.outputs)}
              for p in PRESETS])

## 4. 두 갈래를 섞은 그래프

모멘텀 상위 30(팩터 갈래의 선정)을 **진입 게이트**로 쓰고, 골든크로스(시그널 갈래의 이벤트)가 나면 진입한다.
데드크로스나 10% 손절에 청산한다. 최대 10종목, 진입 비중은 1/10이다.

In [ ]:
graph = example_momentum_gated_cross()
check = validate_strategy(graph)
print("완성 그래프:", check.ok, check.reason or "", check.warnings)
pd.DataFrame([{"순서": i + 1, "노드": n.id, "블록": n.function_id, "역할": get_function(n.function_id).role, "파라미터": n.params}
              for i, n in enumerate(execution_order(graph))])

## 5. 한 시점 실행

`execute`는 연결을 검사하고 위상 순서대로 각 블록을 부른다. 결과는 노드 id와 출력 포트로 꺼낸다.
`event.transition`은 직전 거래일의 조건이 필요해서, 실행기가 그 상류 그래프를 전날 기준으로 한 번 더 돌린다.

In [ ]:
result = execute(graph, ExecutionContext(as_of, history))
for warning in result.warnings:
    print("경고:", warning)
print("유니버스", len(result.port("filter", "universe").symbols))
print("모멘텀 상위 30", [row.instrument_id for row in result.port("leaders", "selection").rows][:10], "…")
print("골든크로스", [row.instrument_id for row in result.port("cross_edge", "entry").rows])
print("데드크로스", [row.instrument_id for row in result.port("cross_edge", "exit").rows])

In [ ]:
decision = result.port("decision", "result")
signals = table(decision)
crossed = {row.instrument_id for row in result.port("cross_edge", "entry").rows}
signals[signals["symbol"].isin(crossed)][["symbol", "eligible", "action", "target_weight", "reason_codes"]]

보유가 없으니 청산은 없다. 골든크로스 종목 중 게이트(상위 30) 안에 있는 것(`eligible` 참)만 `ENTER`가 되고,
게이트 밖 종목은 이벤트가 있어도 `HOLD`다. 기준일을 바꾸면 결과가 달라진다.

## 6. 팩터 갈래 — 목표 포트폴리오에서 판단으로

점수로 고르는 전략은 선정·비중·제약·목표 포트폴리오를 지나 `strategy.state`로 들어간다.

In [ ]:
momentum = execute(example_momentum(), ExecutionContext(as_of, history))
table(momentum.port("target", "portfolio")).assign(cash=momentum.port("target", "portfolio").cash_weight)

## 7. 1년 백테스트

`run_graph_backtest`는 거래일마다 현재 보유를 넣어 그래프를 실행하고, 실행 노드의 정책(리밸런스 주기·이탈 허용·현금 부족 처리)으로
다음 시가에 체결한다. 전략마다 따로 쓴 코드는 없다. 예제 8개를 모두 같은 함수로 돌린다.

In [ ]:
settings = RunSettings(signal_frequency="daily", initial_capital=100_000, fee_rate=0.001, slippage_rate=0.0005)
rows = []
for key, (name, build) in EXAMPLES.items():
    book = run_graph_backtest(build(), history, settings)
    last = book.portfolio_history[-1]
    rows.append({"전략": name, "체결": len(book.trade_log), "미체결": len(book.skipped_orders),
                 "수익률": f"{last.cumulative_return:+.2%}", "최대낙폭": f"{min(p.drawdown for p in book.portfolio_history):+.2%}"})
pd.DataFrame(rows)

## 8. 데이터가 모자랄 때

기본 모멘텀 프리셋(12개월, 최근 1개월 제외)은 종목마다 274개 봉이 필요하다. 저장된 일봉은 251일이라 점수가 비고, 실행기가 경고한다.

In [ ]:
score = place("momentum-score", "mom", {"universe": ("u", "universe")})
probe = StrategyGraph((StrategyNode("u", "universe.compose", {}),) + score.nodes, score.edges)
execute(probe, ExecutionContext(as_of, history)).warnings